# **6. T-Distribution & Small Samples**

## When to Use the T-Distribution
- Sample size is **small** (n < 30)
- Population standard deviation (σ) is **unknown**
- Data is approximately normally distributed

## T-Distribution Properties
- Bell-shaped and symmetric (like normal)
- **Heavier tails** than normal distribution
- Defined by **degrees of freedom** (df = n - 1)
- As df → ∞, t-distribution → normal distribution

## Why Heavier Tails?
With small samples, there's more uncertainty about s (sample std dev), so extreme values are more likely.

In [ ]:
# ===============================================
# T-DISTRIBUTION VS NORMAL DISTRIBUTION
# ===============================================

import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

x = np.linspace(-4, 4, 200)

plt.figure(figsize=(12, 5))

# Normal distribution
plt.plot(x, stats.norm.pdf(x), 'b-', linewidth=2, label='Normal (z)')

# T-distributions with different df
for df, color in [(3, 'red'), (10, 'green'), (30, 'orange')]:
    plt.plot(x, stats.t.pdf(x, df), '--', color=color, linewidth=2, 
             label=f't (df={df})')

plt.xlabel('Value', fontsize=12)
plt.ylabel('Probability Density', fontsize=12)
plt.title('T-Distribution vs Normal Distribution\n(Notice heavier tails for small df)', fontsize=14)
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

print("Key Observations:")
print("  • T-distribution has heavier tails (more probability in extremes)")
print("  • As df increases, t approaches normal")
print("  • At df ≈ 30, practically identical to normal")

In [ ]:
# ===============================================
# CRITICAL VALUES: T VS Z
# ===============================================

from scipy import stats

print("=" * 50)
print("CRITICAL VALUES FOR 95% CONFIDENCE")
print("=" * 50)

# For 95% CI, we need the value that cuts off 2.5% in each tail
confidence = 0.95
tail_prob = (1 - confidence) / 2

# Z critical value (from normal distribution)
z_critical = stats.norm.ppf(1 - tail_prob)
print(f"\nZ-critical (normal): {z_critical:.3f}")

print(f"\nT-critical values for different sample sizes:")
print(f"{'n':>5} {'df':>5} {'t*':>8}")
print("-" * 20)

for n in [5, 10, 15, 20, 30, 50, 100, 1000]:
    df = n - 1
    t_critical = stats.t.ppf(1 - tail_prob, df)
    print(f"{n:>5} {df:>5} {t_critical:>8.3f}")

print(f"\n{'∞':>5} {'∞':>5} {z_critical:>8.3f}  ← Normal")

print("\n💡 Key Insight: Smaller n → larger t* → wider confidence intervals")

In [ ]:
# ===============================================
# SMALL SAMPLE T-TEST EXAMPLE
# ===============================================

import numpy as np
from scipy import stats

print("=" * 50)
print("SMALL SAMPLE T-TEST")
print("=" * 50)

# Scenario: Testing a new drug's effect on blood pressure
# Only 12 patients available for study (small sample!)

np.random.seed(42)
# Blood pressure reduction (mmHg)
reductions = np.array([8, 12, 5, 15, 9, 11, 7, 14, 10, 6, 13, 8])

print(f"\nSample data (BP reduction in mmHg):")
print(f"  {reductions}")
print(f"\nSample size: n = {len(reductions)}")
print(f"Sample mean: x̄ = {np.mean(reductions):.2f}")
print(f"Sample std: s = {np.std(reductions, ddof=1):.2f}")

# Test if drug has ANY effect (μ ≠ 0)
print("\n" + "-" * 50)
print("Hypotheses:")
print("  H₀: μ = 0 (drug has no effect)")
print("  H₁: μ ≠ 0 (drug has some effect)")

t_stat, p_value = stats.ttest_1samp(reductions, 0)

print(f"\nResults:")
print(f"  t-statistic: {t_stat:.3f}")
print(f"  p-value: {p_value:.6f}")

# Confidence interval
n = len(reductions)
se = np.std(reductions, ddof=1) / np.sqrt(n)
ci = stats.t.interval(0.95, df=n-1, loc=np.mean(reductions), scale=se)

print(f"\n95% Confidence Interval: [{ci[0]:.2f}, {ci[1]:.2f}]")

alpha = 0.05
print("\n" + "=" * 50)
if p_value < alpha:
    print(f"✓ SIGNIFICANT! Drug appears to reduce BP.")
else:
    print(f"✗ Not significant.")

In [ ]:
# ===============================================
# PAIRED T-TEST (Before/After)
# ===============================================

import numpy as np
from scipy import stats

print("=" * 50)
print("PAIRED T-TEST (Before/After Design)")
print("=" * 50)

# Scenario: Weight loss program - same people measured before and after

np.random.seed(42)
before = np.array([185, 192, 178, 195, 205, 188, 175, 199, 182, 190])
after = np.array([180, 188, 175, 189, 198, 185, 172, 193, 178, 185])

print(f"\nBefore: {before}")
print(f"After:  {after}")

differences = before - after
print(f"\nWeight lost: {differences}")
print(f"Mean weight lost: {np.mean(differences):.1f} lbs")

print("\n" + "-" * 50)
print("Hypotheses:")
print("  H₀: μ_diff = 0 (no weight loss)")
print("  H₁: μ_diff > 0 (weight was lost)")

# Paired t-test
t_stat, p_value_two = stats.ttest_rel(before, after)
p_value_one = p_value_two / 2  # One-tailed

print(f"\nResults:")
print(f"  t-statistic: {t_stat:.3f}")
print(f"  p-value (one-tailed): {p_value_one:.4f}")

# This is equivalent to one-sample t-test on differences
t_stat2, p_value2 = stats.ttest_1samp(differences, 0)
print(f"\n  (Equivalent one-sample test on differences:")
print(f"   t = {t_stat2:.3f}, p = {p_value2/2:.4f})")

print("\n" + "=" * 50)
if t_stat > 0 and p_value_one < 0.05:
    print("✓ SIGNIFICANT! Program leads to weight loss.")
else:
    print("✗ Not significant.")

---
# **Big Data Considerations**

## The Texas Sharpshooter Fallacy
Drawing targets around bullet holes after shooting - finding "patterns" in random data.

## Big Data Problems

| Issue | Description |
|-------|-------------|
| **Everything is "significant"** | Large n makes tiny effects statistically significant |
| **Multiple testing** | More tests = more false positives |
| **p-hacking** | Running many tests until one is significant |

## Solutions
- Focus on **effect size**, not just p-values
- Use **Bonferroni correction** for multiple tests
- Pre-register hypotheses before analysis
- Prioritize **practical significance**

In [ ]:
# ===============================================
# EFFECT SIZE: COHEN'S D
# ===============================================

import numpy as np
from scipy import stats

print("=" * 50)
print("EFFECT SIZE: Beyond P-Values")
print("=" * 50)

def cohens_d(group1, group2):
    """Calculate Cohen's d for effect size"""
    n1, n2 = len(group1), len(group2)
    var1, var2 = np.var(group1, ddof=1), np.var(group2, ddof=1)
    pooled_std = np.sqrt(((n1-1)*var1 + (n2-1)*var2) / (n1+n2-2))
    return (np.mean(group1) - np.mean(group2)) / pooled_std

np.random.seed(42)

# Scenario: Two samples with TINY difference but huge n
n = 10000
group_a = np.random.normal(100, 15, n)
group_b = np.random.normal(100.5, 15, n)  # Only 0.5 point difference!

t_stat, p_value = stats.ttest_ind(group_a, group_b)
d = cohens_d(group_b, group_a)

print(f"\nLarge Sample Example (n = {n} each):")
print(f"  Mean A: {np.mean(group_a):.2f}")
print(f"  Mean B: {np.mean(group_b):.2f}")
print(f"  Difference: {np.mean(group_b) - np.mean(group_a):.2f}")
print(f"\n  p-value: {p_value:.4f} → {'Significant!' if p_value < 0.05 else 'Not significant'}")
print(f"  Cohen's d: {d:.3f}")

print("\nCohen's d interpretation:")
print("  |d| < 0.2: Negligible")
print("  |d| ≈ 0.2: Small")
print("  |d| ≈ 0.5: Medium")
print("  |d| ≥ 0.8: Large")

print(f"\n⚠️ Despite p < 0.05, effect size is NEGLIGIBLE!")
print("   Statistically significant ≠ Practically important")

---
## Summary

| Concept | Key Point |
|---------|----------|
| **T-Distribution** | Use when n < 30 or σ unknown |
| **Degrees of Freedom** | df = n - 1 |
| **Heavier Tails** | More uncertainty with small samples |
| **Paired T-Test** | For before/after or matched data |
| **Effect Size** | Measures practical significance |
| **Cohen's d** | Small (0.2), Medium (0.5), Large (0.8) |

### Python Reference
```python
from scipy import stats

# T-distribution
stats.t.pdf(x, df)       # PDF
stats.t.cdf(x, df)       # CDF
stats.t.ppf(q, df)       # Inverse CDF

# Tests
stats.ttest_1samp(data, mu)      # One-sample
stats.ttest_ind(g1, g2)          # Independent samples
stats.ttest_rel(before, after)   # Paired
```